# projenın amacı 
projenini amacı  bir şirketin müşterilerinin önceden ayrılıp ayırılmayacağını tahmin etmek 
# tensorflow neyi çözer 
churn() ayrılıp ayrılmayacağının model oluşturarak bize bir tahmin sonuç verir
# kapsanan tensorflow konuları
refrasyon , katmanlar, aktivasyon foksiyonları vb.


In [1]:
import numpy as np 

In [2]:
yas=np.linspace(17,70,100,dtype=int)
aylıkharacma=np.linspace(100,10000,100)
abonesuresı=np.linspace(1,36,100,dtype=int)
destektalep=np.linspace(1,10,100,dtype=int)
X=np.column_stack((yas,aylıkharacma,abonesuresı,destektalep))

In [3]:
churn=0
skor=(yas>50)+(aylıkharacma<4000)+(abonesuresı<3)+(destektalep>2) #boolen oluşturduk true=1 olunana sonuçlar toplanıyor false=0 larda toplanıyor
churn=skor>=2 # yukarıdaki koşullardan iki tanesi true ise churn=2 değeri alır yani müsteri ayrılır
Y=churn.astype(int) # churi Y diye atadık 

In [4]:
N=100
dizi=np.arange(0,N)
np.random.shuffle(dizi)

# Yüzdelik hesapla
train_end=int(0.7*N)       # train eğitim yapacak verinin indeksine kadar belirledik 
validation_end=train_end+int(0.15*N) # validation yapacak verinini indeksine kadar belirledik

# Setleri ayır
trainindex=dizi[0:train_end]
validationindex=dizi[train_end:validation_end]
testindex=dizi[validation_end:]

# setleri oluştur
Xtrain=X[trainindex]
Ytrain=Y[trainindex]

Xval=X[validationindex]
Yval=Y[validationindex]

Xtest=X[testindex]
Ytest=Y[testindex]

In [9]:
# model oluştur
import tensorflow as tf
from tensorflow.keras.models import Sequential # modülü dahil ettik 
from tensorflow.keras.layers import Dense # katmanların dense(hepsi birbirine bağlı şeklinde dahil ettik)

# modeli oluşturduk 
model=Sequential()

# hidden layer 1 (gizli katman 1)                 # shape[1],  virgülü tuple  veri tipinde olsun diye koyduk 
model.add(Dense(64,activation="relu",input_shape=(Xtrain.shape[1],))) # 64 nöron ,aktivasyon fonksiyonu relu,input_shape=verinin özelikleri

# hidden layer 2 (gizli katman 2)
model.add(Dense(32,activation="relu"))#input_shape=(Xtrain.shape[1]bunu vermemize  gerek yok bu kodla veri model ile tanıştı(shape[1]veri özelikler)

# output katmanı (çıktı katmanı)
model.add(Dense(1,activation="sigmoid"))

# model özeti 
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_1 (Dense)                 │ (None, 64)             │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,433 (9.50 KB)

 Trainable params: 2,433 (9.50 KB)

 Non-trainable params: 0 (0.00 B)

# neden sigmoid neden  relu?
relu aktivasyon fonksiyonu negatif değerlerde öğrenmez ama  pozitif değerlerde öğrenir bu sayade hangi nöranların önemli olduğunu öğrenir
karmaşık işlemleri daha hızlı yapabilir 

sigmoid aktivasyonu sonuçta biz churni 1 ve 0 istediğimiz için kulanılırbinary sınıflandırma için kualnılır çıkt olasılık gibi yorumlanabilir
0.84  olasılıkla sınıf 1 gibi 

In [20]:
# model derleme (compile)  # parametreleri burada verdğimiz için eğitimden önce derleme yapılır
model.compile(                                                                    #high loss(yüksek kayıp) gerçeğe ne kadar uzak tahminini gösteririr
    loss="binary_crossentropy",    #(kayıp fonkiyonu)modelin ne kadar hatalı olduğunu ölçer#low loss(düşük kayıp)gerçeğe yakın tahmin yaptıbı gösterir
    
    optimizer="adam",   #hatayı ölçtükten sonra bir dahaki safere hatayı en aza getirmek için ağırları şöyle değiştereceksi der büyükse(başarılı)küçükse(başarısızdır)
    
    metrics=["accuracy"]    # modelin gperformansını ölçer 
)

In [26]:
#model eğitimi
history=model.fit(  # fit çalıştığında aynı zamanda içinde bi history objeside döndiyüryor bu history oobjesinin history e atadık 
    Xtrain,Ytrain, # test verileri
    epochs=12,  # tüm verinin kaç kez modelden geçireleceği 
    batch_size=16,  # her bi parametrede kaç veri deneyeceği
    validation_data=(Xval,Yval) # doğrulama verileri
              
)
history.history.keys() # yani bi objeyi bi değişkene atadığında içindeki sözlüğü kulanabiliyorsun

Epoch 1/12
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 1.0000 - loss: 6.0612e-10 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 2/12
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 1.0000 - loss: 6.0612e-10 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 3/12
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 1.0000 - loss: 6.0612e-10 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 4/12
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 1.0000 - loss: 6.0612e-10 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 5/12
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 1.0000 - loss: 6.0612e-10 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 6/12
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 1.0000 - loss: 6.0612e-10 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 7/12
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 1.0000 - loss: 6.0612e-10 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 8/12
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 1.0000 -

dict_keys(['accuracy', 'loss', 'val_accuracy', 'val_loss'])

In [27]:
# modelin test setinde değerlendirme
test_loss,test_accuracy=model.evaluate(Xtest,Ytest) # evaluate modelin test verisini ölçüyo fit gibi parametreleri değiştirip öğrenme sağlamaz sadece test eder
print(f"Test Loss: {test_loss:.6f}") # hata payını gösterir 6 ondalıklı sayıyla yazıır
print(f"Test Accuracy: {test_accuracy:.4f}") # doğrulunu gösterir 4  ondalıklısayıyıla yazılır

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 1.0000 - loss: 1.2326e-22
Test Loss: 0.000000
Test Accuracy: 1.0000


In [ ]:
# modelde hata payı yok öğrendiği şeyi çok iyi uyguluyor ve overfitting yok 

In [36]:
# Yeni bir müşteri için churn tahmini yapalım
yenimusteri=np.array([[4,5,7,9]]) # modelin bekldeiği input=shape ile aynı olmaalı 4 özlliğimiz vardı iki botyutlu array oluşturduk [[]] bu şeklide 
ytahmin=model.predict(yenimusteri)  # predict eğitilmiş model ile tahmin yapmamızı sağlar
print(f"tahmin değeri{ytahmin}")
esikdegeri=0.5
if esikdegeri < ytahmin:
    print("müşteri ayrılmayacak ")
else:
    print("müşteri büyük ihtimalle ayrılacak ")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
tahmin değeri[[0.25124922]]
müşteri büyük ihtimalle ayrılacak 
